# axon-lang — treinar **um expert por vez** (Colab)

Constrói o expert do pyaxon: `ModularRouter` (roteia a pergunta pra família certa) +
`SparseKB` (índice TF-IDF/LSA que recupera a lição). É o **retrieval**, não um LLM —
o fine-tune do DeepSeek/Qwen fica no outro notebook (`finetune_expert_colab.ipynb`).

**Fluxo:** compila o `_axon.so` → clona só a branch de dados daquele expert →
roda o `build_<x>_experts.py` → mede o roteamento → **salva no Drive**.

Cada expert é independente: treinar `go` não mexe em `rust`. Rode um, salve, troque
o `EXPERT` na célula 5 e rode de novo.

> `Ambiente de execução → Alterar tipo de ambiente → GPU (T4)` (opcional aqui — veja a célula 3).

## 1. Clonar o código (ou atualizar, se já estiver clonado)

In [ ]:
REPO = "https://github.com/geraldogrise/axon-llm.git"

import os, sys
if os.path.isdir("axon-llm"):
    !git -C axon-llm pull --quiet && echo "repo atualizado"
else:
    !git clone --depth 1 $REPO axon-llm

sys.path.insert(0, "/content/axon-llm/notebooks")   # helpers dos notebooks
sys.path.insert(0, "/content/axon-llm/python")      # pacote pyaxon
import axon_colab as ac
ac.tabela()

## 2. Compilar o `_axon.so`

O repo versiona só o `.pyd` do Windows — no Colab a extensão precisa ser compilada.
Leva ~3 min.

In [ ]:
CUDA = False   # True = compila com CUDA/cuBLAS (mais lento de compilar; ver cuda_colab.ipynb)

!apt-get -qq install -y ninja-build > /dev/null
!pip -q install pybind11 numpy

import pybind11, subprocess
cfg = ["cmake", "-S", ".", "-B", "build-colab", "-G", "Ninja",
       "-DCMAKE_BUILD_TYPE=Release",
       "-DAXON_BUILD_PYTHON=ON", "-DAXON_BUILD_TESTS=OFF", "-DAXON_BUILD_EXAMPLES=OFF",
       "-DAXON_ENABLE_NATIVE=OFF",
       f"-DAXON_ENABLE_CUDA={'ON' if CUDA else 'OFF'}",
       f"-DAXON_USE_CUBLAS={'ON' if CUDA else 'OFF'}",
       f"-Dpybind11_DIR={pybind11.get_cmake_dir()}"]
for c in (cfg, ["cmake", "--build", "build-colab", "-j"]):
    p = subprocess.run(c, cwd="axon-llm", capture_output=True, text=True)
    print(p.stdout[-800:] or p.stderr[-800:])
    assert p.returncode == 0, "build falhou (log acima)"

!ls -la axon-llm/python/pyaxon/_axon*.so

## 3. Conferir que o pyaxon importa

In [ ]:
import pyaxon as ax
print("pyaxon ok |", len(ax.tensor([[1.0, 2.0]]).numpy()[0]), "elementos no tensor de teste")
print("cuda_available:", ax.cuda_available())

## 4. Escolher o expert

**Troque só esta linha** pra treinar outro. Um por vez — cada um puxa a própria
branch de dados (`--depth 1`, baixa só o que precisa).

In [ ]:
EXPERT = "go"     # go, rust, python, java, dotnet, js, php, ruby, aws, azure, gcp,
                  # oci, bash, docker, git, kubernetes, shell, web, escolar

branch, sub, env_var, script, rotulo = ac.check(EXPERT)
print(f"expert : {EXPERT}  ({rotulo})")
print(f"dados  : branch {branch} -> {sub}")
print(f"script : examples/{script}  (env {env_var})")
print(f"saida  : examples/axon_lang_data/{ac.SAIDA[EXPERT]}/")

## 5. Treinar e salvar no Drive

`build_expert` clona a branch, aponta a env var pros dados e roda o script. No fim
ele imprime a **acurácia de roteamento** nas perguntas de teste do próprio script.

Os artefatos (`router.*.json` + `kb.sparse.json.gz`) vão pro Drive em
`MyDrive/axon_experts/<expert>_experts/` — a sessão do Colab cai, o Drive não.

In [ ]:
# ajuste se quiser treino mais longo: AXON_EPOCHS (300), AXON_BATCH (256), AXON_LSA_DIM (200)
saida = ac.build_expert(EXPERT, repo="axon-llm", extra_env={"AXON_EPOCHS": "300"})
destino = ac.salvar_expert(saida, EXPERT)

## 6. Testar o expert treinado

Recarrega o que foi salvo e roda as **perguntas de teste do próprio expert** — cada
`build_*_experts.py` traz a sua lista, com a família correta esperada de cada uma. Trocar
o `EXPERT` troca as perguntas junto, então isso funciona igual pros 19.

Pra cada pergunta: `OK` se o router acertou a família, mais o trecho que o KB recuperou.

In [ ]:
import pyaxon as ax

# load() é método de instância e devolve self: cria vazio e carrega por cima.
router = ax.modular.ModularRouter().load(os.path.join(saida, "router"))
kb = ax.vindex.SparseKB().load(os.path.join(saida, "kb.sparse.json.gz"))

testes = ac.perguntas(EXPERT, repo="axon-llm")
ok = 0
for familia, q in testes:
    rota = router.route(q)
    acertou = rota[:1] == [familia]
    ok += acertou
    trechos = kb.retrieve(q, path_prefix=rota, top_k=1)
    snippet = trechos[0][0][:150].replace("\n", " ") if trechos else "(nada)"
    print(f"[{'OK' if acertou else 'X '}] {' > '.join(rota):26} | {q}")
    print(f"        {snippet} ...")

print(f"\nacurácia de família: {ok}/{len(testes)} = {ok / len(testes):.0%}")

## 7. Próximo expert

Volte na **célula 4**, troque o `EXPERT`, e rode 4 → 5 → 6 de novo. Não precisa
recompilar (células 1–3) enquanto a sessão estiver viva.

Quando todos estiverem no Drive, o `ax.system.AxonSystem` carrega a pasta inteira e
roteia **entre** os experts (qual domínio) antes de rotear dentro dele.

```python
sistema = ax.system.AxonSystem.load("/content/drive/MyDrive/axon_experts")
sistema.fit_domain_gate()
sistema.route("como uso goroutines?")   # -> go_experts > concorrencia
```